In [5]:
%pip install -U langchain langchain-community langchain-neo4j langchain-groq sentence-transformers neo4j
%pip install -U langchain-openai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import sys
import getpass
from dotenv import load_dotenv
try:
    import langchain_google_genai
    import langchain_neo4j
    import google.generativeai as genai
except ImportError:
    print("Se instalează pachetele necesare...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "langchain-google-genai", "langchain-neo4j", "neo4j", "google-generativeai"])
    import google.generativeai as genai

from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_google_genai import ChatGoogleGenerativeAI
from config import URI, USER, PASSWORD

print("\nCONECTARE SISTEM...")

load_dotenv()

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
candidate_models = [
    "gemini-1.5-flash",
    "gemini-1.5-flash-latest",
    "gemini-1.5-flash-001",
    "gemini-flash-latest",
    "gemini-1.0-pro",
    "gemini-pro"
]

target_model = None

for model_name in candidate_models:
    try:
        test_llm = ChatGoogleGenerativeAI(model=model_name, temperature=0)
        test_llm.invoke("Hi")
        print("SUCCES!")
        target_model = model_name
        break
    except Exception as e:
        error_msg = str(e)
        if "404" in error_msg or "NOT_FOUND" in error_msg:
            pass
        elif "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
            pass
        else:
            pass

if not target_model:
    print("\nEROARE CRITICĂ: Niciun model Gemini nu răspunde.")
    print("Verifică dacă ai activat API-ul în Google Console sau dacă ai Billing activat (unele modele cer card).")
    sys.exit()

print(f"\nAM SELECTAT: '{target_model}'")
#llm
try:
    llm = ChatGoogleGenerativeAI(
        model=target_model,
        temperature=0,
        convert_system_message_to_human=True
    )
except Exception as e:
    print(f"Eroare fatală la inițializare LLM: {e}")
    sys.exit()

#baza de date
try:
    graph = Neo4jGraph(url=URI, username=USER, password=PASSWORD)
    graph.refresh_schema()
    print("Conectat la Neo4j.")
except Exception as e:
    print(f"Eroare conectare Neo4j: {e}")
    sys.exit()

try:
    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=False,
        allow_dangerous_requests=True,
        validate_cypher=True
    )
except Exception as e:
    print(f"Eroare creare lanț: {e}")
    sys.exit()

#  CHAT
print("\n" + "="*60)
print(f"AGENT ACTIVAT (Model: {target_model})")
print("="*60)

while True:
    try:
        q = input("\nÎntrebare: ")
    except EOFError: break
    if q.lower() in ['exit', 'stop']: break
    if not q.strip(): continue

    try:
        res = chain.invoke({"query": q})
        print("\n" + "-"*40)
        print(f"RĂSPUNS: {res['result']}")
        print("-" * 40)
    except Exception as e:
        print(f"\nEroare: {e}")


CONECTARE SISTEM...
SUCCES!

AM SELECTAT: 'gemini-flash-latest'
Conectat la Neo4j.

AGENT ACTIVAT (Model: gemini-flash-latest)

----------------------------------------
RĂSPUNS: Nu știu răspunsul.
----------------------------------------

----------------------------------------
RĂSPUNS: Certificările disponibile sunt: AWS Solutions Architect, CKA (Kubernetes), și PMP.
----------------------------------------

----------------------------------------
RĂSPUNS: Nu știu răspunsul.
----------------------------------------

----------------------------------------
RĂSPUNS: Nu știu răspunsul.
----------------------------------------

----------------------------------------
RĂSPUNS: Ana Popa, Paul Serban lucrează sau a lucrat la Google.
----------------------------------------


KeyboardInterrupt: Interrupted by user